# Get the data needed for DeepSTARR 
to do: </br>
    - other data augmentation </br>
    - other normalisation method, see below </br>
Done:  </br> 
    - Try with log normalisation --> Tricky to evaluate, the base pair scores look better, but the overall correlations are slightly lower. Maybe would be improved by performing the same normalisation but without the log transformation, and scaling by a factor that gets numbers in the 0 - 5 range </br> 

In [2]:
here::i_am("atac/archR/DeepSTARR/trial1_Export_data.ipynb")

#####################
## Define settings ##
#####################

source(here::here("settings.R"))
source(here::here("utils.R"))
suppressPackageStartupMessages({
    library(BSgenome.Mmusculus.UCSC.mm10)
    library(parallel)
})

here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/atlasses/gastrulation_multiome/code



In [3]:
# Options
args = list()
args$pseudobulk_matrix = file.path(io$basedir, 'results/atac/archR/pseudobulk/celltype/PeakMatrix/pseudobulk_PeakMatrix_summarized_experiment.rds')
args$celltypes = c("Epiblast",
                    "Primitive_Streak" ,
                    "Nascent_mesoderm",
                    "ExE_mesoderm",
                    "Mixed_mesoderm",
                    "Allantois",
                    "Mesenchyme",
                    "Haematoendothelial_progenitors",
                    "Endothelium",
                    "Blood_progenitors_1",
                    "Blood_progenitors_2",
                    "Erythroid1",
                    "Erythroid2",
                    "Erythroid3")
args$markers = file.path(io$basedir, 'results/atac/archR/differential/pseudobulk/celltype/PeakMatrix/parsed/markers_filt.txt.gz')
args$outdir = file.path(io$basedir, 'results/atac/archR/DeepSTARR/epiblast_blood')
dir.create(args$outdir, recursive=T, showWarnings=F)

In [5]:
# Load files
Peakmatrix = readRDS(args$pseudobulk_matrix)

In [9]:
head(Peakmatrix)

class: SummarizedExperiment 
dim: 6 37 
metadata(0):
assays(1): PeakMatrix
rownames(6): chr1:3035602-3036202 chr1:3062653-3063253 ...
  chr1:3340575-3341175 chr1:3443943-3444543
rowData names(4): seqnames idx start end
colnames(37): Allantois Anterior_Primitive_Streak ... Surface_ectoderm
  Visceral_endoderm
colData names(27): TSSEnrichment ReadsInTSS ... FRIP nCells

In [10]:
Peakmatrix_filt = Peakmatrix[,args$celltypes]
Peakmatrix_filt

class: SummarizedExperiment 
dim: 192251 14 
metadata(0):
assays(1): PeakMatrix
rownames(192251): chr1:3035602-3036202 chr1:3062653-3063253 ...
  chrX:169925487-169926087 chrX:169937064-169937664
rowData names(4): seqnames idx start end
colnames(14): Epiblast Primitive_Streak ... Erythroid2 Erythroid3
colData names(27): TSSEnrichment ReadsInTSS ... FRIP nCells

In [11]:
# Split data in train, val, and test sets
# Before did it with the sample function, but this results in different peaks every time
# all_peaks = rownames(Peakmatrix_filt)
# paste0('total peaks: ', length(all_peaks))
# tot_peaks = length(all_peaks)
# val_peaks = sample(all_peaks, tot_peaks/12)
# paste0('peaks in val set: ', length(val_peaks))
# train_peaks = all_peaks[!all_peaks%in%val_peaks]
# paste0('peaks without validation set: ', length(train_peaks))
# test_peaks = sample(train_peaks, tot_peaks/12)
# paste0('peaks in test set: ', length(test_peaks))
# train_peaks = train_peaks[!train_peaks%in%test_peaks]
# paste0('peaks without validation & test set: ', length(train_peaks))

# Train and validation are around 1/10th of train set

all_peaks = rownames(Peakmatrix_filt)
paste0('total peaks: ', length(all_peaks))
tot_peaks = length(all_peaks)
val_test = tot_peaks/12
val_peaks = all_peaks[1:val_test]
paste0('peaks in val set: ', length(val_peaks))
train_peaks = all_peaks[!all_peaks%in%val_peaks]
paste0('peaks without validation set: ', length(train_peaks))
test_peaks = all_peaks[val_test+1:val_test*2]
paste0('peaks in test set: ', length(test_peaks))
train_peaks = train_peaks[!train_peaks%in%test_peaks]
paste0('peaks without validation & test set: ', length(train_peaks))

[1] "total peaks: 192251"

[1] "peaks in val set: 16020"

[1] "peaks without validation set: 176231"

[1] "peaks in test set: 16020"

[1] "peaks without validation & test set: 160211"

### Define functions

In [34]:
# transform peaks to 249 bp flanking centre
# Little bit different than DeepSTARR paper, where they also take regions flanking the centre, but would need to recalculate the accessibility of the new bins
parse_peaks = function(peaks){
    data.table(peak=peaks,
               chr=strsplit(peaks, ':') %>% map_chr(1),
               start=as.numeric(strsplit(strsplit(peaks, ':') %>% map_chr(2), '-') %>% map_chr(1)),
               end=as.numeric(strsplit(strsplit(peaks, ':') %>% map_chr(2), '-') %>% map_chr(2))) %>%
    .[,centre:=round((start+end)/2, 0)] %>% # Get centre of peak and round in case original peak width was an odd number
    .[,`:=`(start=NULL,end=NULL)] %>%
    .[,`:=`(start = centre - 124,
            end = centre + 124)] %>%
    .[,new_peak:=paste0(chr, ':', start, '-', end)]
}

# prepare fasta file
get_fa = function(x, peakset=train_peaks){ # x=row
    # positive strand
    pos_peak = paste0('>', peakset[x,new_peak], '_+')
    pos_fa = paste(as.character(as.vector(getSeq(Mmusculus, peakset[x,chr],
                                         start=peakset[x,start], end=peakset[x,end]))), collapse="")
    # negative strand
    neg_peak = paste0('>', peakset[x,new_peak], '_-')
    neg_fa = paste(as.character(as.vector(reverseComplement(DNAString(pos_fa)))), collapse="")
    
    # return
    tmp = c(pos_peak, pos_fa, neg_peak, neg_fa)
    return(tmp)
}

## Get fasta files

In [13]:
########################
### Get & save Fastas ##
########################

# Train set
train_peakset = parse_peaks(train_peaks)
train_fa = mclapply(1:nrow(train_peakset), get_fa, peakset=train_peakset, mc.cores=24) %>% unlist()
writeLines(train_fa , sprintf('%s/train.fa', args$outdir))

In [ ]:
# Validation set
val_peakset = parse_peaks(val_peaks)
val_fa = mclapply(1:nrow(val_peakset), get_fa, peakset=val_peakset, mc.cores=24) %>% unlist()
writeLines(val_fa , sprintf('%s/val.fa', args$outdir))

In [ ]:
# Test set
test_peakset = parse_peaks(test_peaks)
test_fa = mclapply(1:nrow(test_peakset), get_fa, peakset=test_peakset, mc.cores=24) %>% unlist()
writeLines(test_fa , sprintf('%s/test.fa', args$outdir))

## Get scores

#### Mean accessibility (acc divided by cell numbers)
- This method does not take read count into consideration, see if log2 normalised (below) works better!

In [16]:
# Calculating mean by dividing by number of cells in cell-type
mean_accessibility = function(object){
    counts.mtx = object@assays@data$PeakMatrix
    counts_mean.mtx = lapply(colnames(object), function(x){
        tmp = counts.mtx[,x]/as.data.table(colData(object), keep.rownames=T)[rn==x, nCells]
        
        return(as.data.table(tmp) %>% setnames(x))
    }) %>% do.call('cbind', .) %>% as.matrix()
    rownames(counts_mean.mtx) = rownames(object)
    return(counts_mean.mtx)
}

In [17]:
normalised = mean_accessibility(Peakmatrix_filt)

Warning message in .local(x, row.names, optional, ...):
“Arguments in '...' ignored”
Warning message in .local(x, row.names, optional, ...):
“Arguments in '...' ignored”
Warning message in .local(x, row.names, optional, ...):
“Arguments in '...' ignored”
Warning message in .local(x, row.names, optional, ...):
“Arguments in '...' ignored”
Warning message in .local(x, row.names, optional, ...):
“Arguments in '...' ignored”
Warning message in .local(x, row.names, optional, ...):
“Arguments in '...' ignored”
Warning message in .local(x, row.names, optional, ...):
“Arguments in '...' ignored”
Warning message in .local(x, row.names, optional, ...):
“Arguments in '...' ignored”
Warning message in .local(x, row.names, optional, ...):
“Arguments in '...' ignored”
Warning message in .local(x, row.names, optional, ...):
“Arguments in '...' ignored”
Warning message in .local(x, row.names, optional, ...):
“Arguments in '...' ignored”
Warning message in .local(x, row.names, optional, ...):
“Argument

In [18]:
##################
### Save scores ##
##################
# scores need to be duplicated, since we have positive and negative strand in .fa file
# Train set
train_sequence_activity = round(normalised[rep(train_peakset$peak, each=2),], 8)
fwrite(train_sequence_activity,  sprintf('%s/train_sequence_activity.txt', args$outdir), sep = '\t')

# Validation set
val_sequence_activity = round(normalised[rep(val_peakset$peak, each=2),], 8)
fwrite(val_sequence_activity,  sprintf('%s/val_sequence_activity.txt', args$outdir), sep = '\t')

# Test set
test_sequence_activity = round(normalised[rep(test_peakset$peak, each=2),], 8)
fwrite(test_sequence_activity,  sprintf('%s/test_sequence_activity.txt', args$outdir), sep = '\t')

x being coerced from class: matrix to data.table



ERROR: Error in eval(expr, envir, enclos): object 'val_peakset' not found


#### log2 normalisation

In [19]:
assayNames(Peakmatrix_filt)[1] <- "counts"
Peakmatrix_filt@assays@data$logcounts <- log2(1e6*(sweep(Peakmatrix_filt@assays@data$counts,2,colSums(Peakmatrix_filt@assays@data$counts),"/"))+1)

In [20]:
lognormalised = Peakmatrix_filt@assays@data$logcounts

In [21]:
rownames(lognormalised) = rownames(Peakmatrix_filt)

In [22]:
# get peak sets (after reloading notebook)
train_peakset = parse_peaks(train_peaks)
val_peakset = parse_peaks(val_peaks)
test_peakset = parse_peaks(test_peaks)

In [23]:
##################
### Save scores ##
##################
# scores need to be duplicated, since we have positive and negative strand in .fa file
# Train set
train_sequence_activity = round(lognormalised[rep(train_peakset$peak, each=2),], 8)
fwrite(train_sequence_activity,  sprintf('%s/train_sequence_activity_log.txt', args$outdir), sep = '\t')

# Validation set
val_sequence_activity = round(lognormalised[rep(val_peakset$peak, each=2),], 8)
fwrite(val_sequence_activity,  sprintf('%s/val_sequence_activity_log.txt', args$outdir), sep = '\t')

# Test set
test_sequence_activity = round(lognormalised[rep(test_peakset$peak, each=2),], 8)
fwrite(test_sequence_activity,  sprintf('%s/test_sequence_activity_log.txt', args$outdir), sep = '\t')

x being coerced from class: matrix to data.table

x being coerced from class: matrix to data.table

x being coerced from class: matrix to data.table



#### Inspection fastas

In [9]:
################################
### Get Fastas for inspection ##
################################
# prepare fasta file
get_fa_ss = function(x, peakset=train_peaks){ # x=row
    # positive strand
    pos_peak = paste0('>', peakset[x,new_peak], '_+')
    pos_fa = paste(as.character(as.vector(getSeq(Mmusculus, peakset[x,chr],
                                         start=peakset[x,start], end=peakset[x,end]))), collapse="")
    # return
    tmp = c(pos_peak, pos_fa)
    return(tmp)
}

##### Marker peaks from Ricard

In [24]:
marker_peaks = fread(args$markers)

In [25]:
marker_peaks_filt = marker_peaks[celltype %in% args$celltypes] %>% 
    .[order(-logFC)]

In [27]:
get_markers_fa = function(ct, n=200){
    peaks_tmp = parse_peaks(marker_peaks_filt[celltype==ct, feature] %>% head(n))
    fa_tmp = mclapply(1:nrow(peaks_tmp), get_fa_ss, peakset=peaks_tmp, mc.cores=24) %>% unlist()
    writeLines(fa_tmp , sprintf('%s/%s_inspection_markers.fa', args$outdir, ct))    
}

In [29]:
for(i in args$celltypes){
    get_markers_fa(i, n = 200)
}

#### peaks from Luke

In [76]:
get_markers_fa = function(peaks, name, n=2000, cores=24){
    peaks_tmp = parse_peaks(peaks) %>% head(n)
    fa_tmp = mclapply(1:nrow(peaks_tmp), get_fa_ss, peakset=peaks_tmp, mc.cores=cores) %>% unlist()
    fa_tmp = fa_tmp[grep('Error',fa_tmp, invert =T)] # for some reason sometimes throws errors so remove as tmp fix
    writeLines(fa_tmp , sprintf('%s/%s.fa', args$outdir, name))    
}

In [13]:
# SCL in FLK+ cells
SCL_FLK = fread(sprintf('%s/SCL_Flk1+_mm10.bed', args$outdir))

In [77]:
get_markers_fa(SCL_FLK$V4, name='SCL_FLK', n=2000, cores=24) 

Warning message in mclapply(1:nrow(peaks_tmp), get_fa_ss, peakset = peaks_tmp, mc.cores = cores):
“scheduled cores 15, 16, 17, 18, 19 encountered errors in user code, all values of the jobs will be affected”
